# Data Validation & Comparison Notebook

Original (`data/Table.csv`, n=96) vs. synthetic (`synthetic/synthetic_crystal_growth_dataset.csv`, n=2000) data.

This notebook reads the outputs already produced by `01_dataset_analysis.py` .. `06_ml_evaluation.py` -- it does not regenerate data or refit models, so it runs in seconds. Re-run those scripts first (see `08_REPRODUCIBILITY_INSTRUCTIONS.md`) if you want fresh numbers.

**Statistical similarity shown here is not evidence of physical validity** -- see `05_VALIDATION_REPORT.md` and `07_ASSUMPTIONS_AND_LIMITATIONS.md` for the full, honest scope of these results.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import Image, display

ROOT = Path('..').resolve() if Path('data').exists() is False else Path('.').resolve()
SYN = Path('.').resolve()
original = pd.read_csv(SYN.parent / 'data' / 'Table.csv')
synthetic = pd.read_csv(SYN / 'synthetic_crystal_growth_dataset.csv')
print('original:', original.shape, ' synthetic:', synthetic.shape)

## 1. Original dataset analysis (recap)

Full output: `results/dataset_analysis.json`.

In [ ]:
with open('results/dataset_analysis.json') as f:
    analysis = json.load(f)
print('Shape:', analysis['shape'])
print('Duplicate rows:', analysis['duplicate_rows'])
print('Ordering constraint violations:', analysis['ordering_constraint_T1_to_T5_monotone_nonincreasing']['n_violations'])
print('Derived-variable identities:', analysis['derived_variable_identities'])

## 2. Kolmogorov–Smirnov and Wasserstein-distance comparison

Full output: `results/ks_wasserstein_tests.csv`.

In [ ]:
ks_df = pd.read_csv('results/ks_wasserstein_tests.csv')
ks_df

## 3. Correlation-structure comparison

Full output: `results/correlation_comparison.csv`. Largest divergences shown first (all involve `D (m)` -- see `05_VALIDATION_REPORT.md` §2 for discussion).

In [ ]:
corr_df = pd.read_csv('results/correlation_comparison.csv')
corr_df.head(10)

In [ ]:
display(Image(filename='plots/correlation_comparison.png'))

## 4. Distribution comparison (example: Lc and D)

All columns are plotted individually in `plots/distribution_comparison_<col>.png`.

In [ ]:
display(Image(filename='plots/distribution_comparison_Lc_m.png'))

In [ ]:
display(Image(filename='plots/distribution_comparison_D_m.png'))

## 5. Pair plot (Lc, Le, D)

In [ ]:
display(Image(filename='plots/pairplot_comparison.png'))

## 6. Duplicate / near-duplicate and outlier check

Recomputed directly here (cheap) rather than re-reading a saved report, for a self-contained sanity check.

In [ ]:
input_cols = ['T1 (K)', 'T2 (K)', 'T3 (K)', 'T4 (K)', 'T5 (K)']
exact_dup = synthetic[input_cols].duplicated().sum()
print('Exact duplicate synthetic input rows:', exact_dup)

ref = original[input_cols].values
ref_std = ref.std(axis=0); ref_std[ref_std == 0] = 1.0
syn_scaled = synthetic[input_cols].values / ref_std
ref_scaled = ref / ref_std
min_dists = np.array([np.sqrt(((ref_scaled - row) ** 2).sum(axis=1)).min() for row in syn_scaled])
print('Synthetic rows within 0.05 normalized distance of an original row:', int((min_dists < 0.05).sum()))

## 7. Predictive-relationship (linear coefficient sign) comparison

Full output: `results/predictive_relationship_comparison.csv`.

In [ ]:
pred_df = pd.read_csv('results/predictive_relationship_comparison.csv')
print(f"Sign agreement: {pred_df['same_sign'].sum()}/{len(pred_df)}")
pred_df

## 8. ML evaluation: original-only vs. synthetic-augmented (CV summary)

Full outputs: `results/cv_results_summary.csv`, `results/final_test_results.csv`. See `06_MODEL_EVALUATION_REPORT.md` for the full narrative -- headline finding: augmentation does NOT improve the single best model (GaussianProcess) on the untouched original test set, though it helps weaker models (e.g. RandomForest) and can hurt others (SVR) substantially.

In [ ]:
cv_summary = pd.read_csv('results/cv_results_summary.csv')
cv_summary.sort_values(['target', 'regime', 'R2_mean'], ascending=[True, True, False])

In [ ]:
final_df = pd.read_csv('results/final_test_results.csv')
final_df.sort_values(['target', 'regime', 'R2'], ascending=[True, True, False])

## 9. GPR prediction-interval coverage (uncertainty-aware model)

Full output: `results/gpr_prediction_intervals.csv`.

In [ ]:
pd.read_csv('results/gpr_prediction_intervals.csv')

## Summary

This notebook reproduces, interactively, the same validation and evaluation results narrated in `05_VALIDATION_REPORT.md` and `06_MODEL_EVALUATION_REPORT.md`. See `07_ASSUMPTIONS_AND_LIMITATIONS.md` for what these results do and do not establish.